In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalog", "catalog_project")
dbutils.widgets.text("schema", "bronze")
dbutils.widgets.text("storage-account", "adlssmartdata1921")

In [0]:
container = dbutils.widgets.get("container")
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
storage_account = dbutils.widgets.get("storage-account")

file_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/producto.json"


In [0]:

productschema = StructType(fields=[
    StructField("idProducto", IntegerType(), False),
    StructField("descripcion", StringType(), False)
])

In [0]:

dfproduct = spark.read \
    .schema(productschema) \
    .option("multiLine", True) \
    .json(file_path)

In [0]:
dfproductdate = dfproduct.withColumn("ingestion_date", current_timestamp())

In [0]:
product_df = dfproductdate.select(
    "idProducto",
    "descripcion",
    "ingestion_date"
)


In [0]:
product_df.write \
    .mode("overwrite") \
    .insertInto(f"{catalog}.{schema}.product")